# Vanilla GCN Fundamentals

**Reference:** William L. Hamilton — *Graph Representation Learning* (2020), Chapters 5 & 7.

This notebook implements a **Vanilla Graph Convolutional Network (GCN)** from scratch — step by step — using only PyTorch, NumPy, NetworkX, and Matplotlib. No PyTorch Geometric. No DGL.

The core equation we implement:
$$
H^{(k)} = \sigma\!\left(\tilde{A}\, H^{(k-1)}\, W^{(k)}\right)
$$
where $\tilde{A} = \hat{D}^{-1/2}\,\hat{A}\,\hat{D}^{-1/2}$,  $\hat{A} = A + I$.

## 1. Imports and Configuration

In [ ]:
import sys, pathlib
# Allow notebook to find the src/ package
ROOT = pathlib.Path.cwd().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# vanilla_gcn package
from vanilla_gcn.config import load_config
from vanilla_gcn.seed import set_seed
from vanilla_gcn.data.synthetic import create_synthetic_graph
from vanilla_gcn.data.preprocessing import (
    add_self_loops, compute_degree_matrix,
    compute_inverse_sqrt_degree, symmetric_normalize, prepare_graph
)
from vanilla_gcn.models.gcn_layer import GCNLayer
from vanilla_gcn.models.vanilla_gcn import VanillaGCN
from vanilla_gcn.training.trainer import train_gcn
from vanilla_gcn.training.evaluation import evaluate_gcn
from vanilla_gcn.visualization.graph import visualize_graph, visualize_predictions
from vanilla_gcn.visualization.embeddings import visualize_embeddings, visualize_training_curves
from vanilla_gcn.visualization.layers import visualize_layer_embeddings
from vanilla_gcn.analysis.receptive_field import compute_k_hop_neighbors, visualize_receptive_field
from vanilla_gcn.utils.logging import setup_logging

setup_logging('INFO', force=True)
cfg = load_config(ROOT / 'configs' / 'default.yaml')
set_seed(cfg.seed)
print('Config loaded. Seed:', cfg.seed)

## 2. Synthetic Dataset

We use a **12-node graph with 3 communities** (classes). Each community has 4 densely connected nodes with sparse inter-community bridges. Node features are drawn from class-specific Gaussians so the GCN can learn to separate them.

In [ ]:
data = create_synthetic_graph(
    seed=cfg.seed,
    train_ratio=cfg.data.train_ratio,
    val_ratio=cfg.data.validation_ratio,
    test_ratio=cfg.data.test_ratio,
)
print(data)

## 3. Original Graph Visualization — BEFORE GCN

Nodes coloured by **true class label**.

In [ ]:
fig = visualize_graph(
    data.adjacency, data.labels, data.node_ids,
    title='BEFORE GCN — Original Graph (True Labels)',
    save_path=ROOT / 'outputs/figures/00_original_graph.png',
)
plt.show()

## 4. Mathematical Formulation

### The full pipeline:
```
H^(0) = X                              ← raw features

Â     = A + I                          ← self-loops

D̂_ii  = Σ_j Â_ij                      ← degree of Â

Ã     = D̂^(-1/2) Â D̂^(-1/2)          ← symmetric normalization

H^(k) = σ( Ã H^(k-1) W^(k) )         ← GCN layer
```

## 5. Step 1 — Add Self-Loops: Â = A + I

In [ ]:
A = data.adjacency
print('A (raw adjacency):')
print(A.numpy())
print(f'Shape: {tuple(A.shape)}')

A_hat = add_self_loops(A)
print('\nÂ = A + I (self-loops added):')
print(A_hat.numpy())
print('Diagonal (all should be 1):', A_hat.diagonal().numpy())

## 6. Step 2 — Degree Matrix D̂

In [ ]:
D_hat = compute_degree_matrix(A_hat)
print('D̂ diagonal (degree of each node in Â):')
print(D_hat.diagonal().numpy())

## 7. Step 3 — Symmetric Normalization: Ã = D̂^(-1/2) Â D̂^(-1/2)

In [ ]:
D_inv_sqrt = compute_inverse_sqrt_degree(D_hat)
print('D̂^(-1/2) diagonal:')
print(D_inv_sqrt.diagonal().numpy())

A_tilde = symmetric_normalize(A_hat)
print('\nÃ (symmetrically normalized adjacency):')
print(np.round(A_tilde.numpy(), 4))
print(f'\nÃ shape: {tuple(A_tilde.shape)}')
print(f'Value range: [{A_tilde.min():.4f}, {A_tilde.max():.4f}]')
print(f'Symmetric? {torch.allclose(A_tilde, A_tilde.T, atol=1e-6)}')

## 8. Manual Single-Node GCN Calculation

Let's manually compute **one forward pass** for **node 0** to see exactly what the GCN does.

For node $u=0$ at layer $k$:
$$
h_u^{(k)} = \sigma\!\left( W^{(k)\top} \sum_{v \in \mathcal{N}(u) \cup \{u\}} \alpha_{uv}\, h_v^{(k-1)} \right)
$$
where $\alpha_{uv} = \tilde{A}_{uv}$.

In [ ]:
TARGET_NODE = 0
X = data.features
y = data.labels

# Neighbours of node 0 in Â (including itself)
a_row_0 = A_hat[TARGET_NODE]   # row of Â
neighbours = torch.where(a_row_0 > 0)[0].tolist()
print(f'Node {TARGET_NODE} neighbours (incl. self) in Â: {neighbours}')

# Normalisation weights α_0v from Ã
print('\nNormalisation weights α_0v = Ã[0, v]:')
for v in neighbours:
    print(f'  α_{{0,{v}}} = {A_tilde[TARGET_NODE, v]:.4f}')

# Weighted aggregation: Σ_v α_0v h_v^(0)
aggregated = torch.zeros(data.num_features)
for v in neighbours:
    aggregated += A_tilde[TARGET_NODE, v] * X[v]

print(f'\nRaw feature h_0^(0): {X[TARGET_NODE].numpy()}')
print(f'Aggregated   Σ α_0v h_v^(0): {aggregated.numpy()}')
print('\n→ This aggregated vector is then multiplied by W^(1) and passed through σ.')
print('  Equivalently: row 0 of (Ã @ X) gives the same result.')
print(f'  Ã[0,:] @ X = {(A_tilde[TARGET_NODE] @ X).numpy()}')
print(f'  Match: {torch.allclose(aggregated, A_tilde[TARGET_NODE] @ X, atol=1e-6)}')

## 9. GCN Layer

The `GCNLayer` implements **exactly** `H_out = σ(Ã @ H @ W)`.

In [ ]:
set_seed(cfg.seed)
layer = GCNLayer(
    input_dim=data.num_features,
    output_dim=16,
    activation=torch.relu,
    debug=True,
)
print('GCN Layer:', layer)
print('W shape:', layer.weight.shape)
print('Trainable:', layer.weight.requires_grad)

# One forward pass
H1 = layer(X, A_tilde)
print('\nH^(1) shape:', H1.shape)
print('H^(1)[node 0]:', H1[0].detach().numpy())

## 10. Full VanillaGCN Model

In [ ]:
set_seed(cfg.seed)
_, A_tilde, X, y = prepare_graph(data.adjacency, data.features, data.labels)

model = VanillaGCN(
    input_dim=data.num_features,
    hidden_dim=cfg.model.hidden_dim,
    num_classes=data.num_classes,
    num_layers=cfg.model.num_layers,
    dropout=cfg.model.dropout,
    debug=cfg.debug.enabled,
)
model.print_summary()

## 11. Forward Pass — Intermediate Representations

In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(X, A_tilde, return_intermediate=True)

print('Intermediate representations:')
for key, tensor in outputs.items():
    print(f'  {key:12s}  shape={tuple(tensor.shape)}  '
          f'min={tensor.min():.3f}  max={tensor.max():.3f}')

## 12. Training

In [ ]:
set_seed(cfg.seed)
model = VanillaGCN(
    input_dim=data.num_features,
    hidden_dim=cfg.model.hidden_dim,
    num_classes=data.num_classes,
    num_layers=cfg.model.num_layers,
    debug=False,
)

history = train_gcn(
    model, A_tilde, X, y,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    epochs=cfg.training.epochs,
    lr=cfg.training.learning_rate,
    weight_decay=cfg.training.weight_decay,
)
print(history.summary())

## 13. Training Curves

In [ ]:
fig = visualize_training_curves(
    history.train_loss, history.val_loss,
    history.train_acc, history.val_acc,
    history.epochs,
    save_path=ROOT / 'outputs/figures/01_training_curves.png',
)
plt.show()

## 14. Evaluation

In [ ]:
metrics = evaluate_gcn(
    model, A_tilde, X, y,
    data.train_mask, data.val_mask, data.test_mask,
)
print(metrics.summary())

## 15. Intermediate Embeddings Visualization

We inspect $X$, $H^{(1)}$, and logits via PCA projection.

In [ ]:
embeddings = model.get_embeddings(X, A_tilde)

fig = visualize_layer_embeddings(
    embeddings, data.labels, data.node_ids,
    save_path=ROOT / 'outputs/figures/01_layer_embeddings.png',
)
plt.show()

## 16. FINAL GRAPH — AFTER GCN

Compare **BEFORE GCN** (true labels) with **AFTER GCN** (predicted labels).
Incorrectly predicted nodes are highlighted with a **red border**.

In [ ]:
pred_labels = metrics.predictions

print('Node-level results:')
print(f'{"Node":>5}  {"True":>5}  {"Pred":>5}  {"Correct":>7}')
for i in data.node_ids:
    correct = '✓' if data.labels[i] == pred_labels[i] else '✗'
    print(f'{i:>5}  {data.labels[i].item():>5}  {pred_labels[i].item():>5}  {correct:>7}')

n_correct = (data.labels == pred_labels).sum().item()
print(f'\nOverall: {n_correct}/{data.num_nodes} correct ({100*n_correct/data.num_nodes:.1f}%)')

In [ ]:
fig = visualize_predictions(
    data.adjacency, data.labels, pred_labels, data.node_ids,
    title='BEFORE GCN vs AFTER GCN',
    save_path=ROOT / 'outputs/figures/01_final_predictions.png',
)
plt.show()

## 17. Receptive Field Analysis

In [ ]:
for node in [0, 4]:
    hop_sets = compute_k_hop_neighbors(data.adjacency, node, max_k=3)
    print(f'Node {node} receptive field:')
    for k, s in hop_sets.items():
        print(f'  {k}-hop: {sorted(s)}')
    print()

fig = visualize_receptive_field(
    data.adjacency, node=0, labels=data.labels, node_ids=data.node_ids,
    max_k=cfg.model.num_layers,
    save_path=ROOT / 'outputs/figures/01_receptive_field.png',
)
plt.show()

## 18. Final Mathematical Summary

The complete pipeline we just executed:
```
H^(0) = X                              (N × F input features)
Â     = A + I                          (N × N, self-loops)
D̂_ii  = Σ_j Â_ij                      (degree matrix)
Ã     = D̂^(-1/2) Â D̂^(-1/2)          (symmetric normalization)

H^(1) = ReLU( Ã X  W^(1) )           (N × hidden_dim)
H^(2) = ReLU( Ã H^(1) W^(2) )        (N × hidden_dim)
logits = Ã H^(2) W^(K)               (N × num_classes)

ŷ_u   = argmax_c logits_{u,c}
```

## 19. How to Replace the Synthetic Dataset

```python
from vanilla_gcn.data.loader import load_graph_from_numpy
import numpy as np

A = np.load('data/raw/my_adjacency.npy')  # (N, N)
X = np.load('data/raw/my_features.npy')   # (N, F)
y = np.load('data/raw/my_labels.npy')     # (N,)

data = load_graph_from_numpy(A, X, y, name='my_dataset')
# Then use exactly the same model, training, and evaluation code above.
```

See `docs/dataset_interface.md` for full details.